# SC-13-Fuzz-Invariants - Fuzz Testing

[<< Foundry Testing](SC-12-Foundry-Testing.ipynb) | [Formal Verification >>](SC-14-Formal-Verification.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre le **fuzz testing** et ses avantages
2. Ecrire des tests avec **parametres aleatoires**
3. Tester des **invariants** de contrats
4. Utiliser `vm.assume` pour filtrer les entrees

### Prerequis

- [SC-12-Foundry-Testing](SC-12-Foundry-Testing.ipynb) complete
- Foundry installe et projet initialise
- Notions de tests unitaires en Solidity

### Duree estimee : 40 minutes

---

## 1. Introduction au Fuzz Testing

Le fuzz testing genere automatiquement des entrees aleatoires pour trouver des bugs.

In [1]:
# Concept de fuzz testing
print("""
FUZZ TESTING - Concepts cles

1. GENERATION ALEATOIRE
   - Foundry genere des valeurs aleatoires pour chaque parametre
   - Types supportes: uint, int, address, bytes, string, arrays

2. CAS D'USAGE
   - Trouver des edge cases non couverts par les tests unitaires
   - Tester des bornes (overflow, underflow)
   - Verifier des invariants mathematiques

3. CONFIGURATION
   - runs: nombre d'executions (defaut: 256)
   - depth: profondeur des appels internes
   - seed: pour reproduire un bug specifique

4. COMMANDE
   forge test --fuzz-runs 1000 --match-path test/Fuzz
""")


FUZZ TESTING - Concepts cles

1. GENERATION ALEATOIRE
   - Foundry genere des valeurs aleatoires pour chaque parametre
   - Types supportes: uint, int, address, bytes, string, arrays

2. CAS D'USAGE
   - Trouver des edge cases non couverts par les tests unitaires
   - Tester des bornes (overflow, underflow)
   - Verifier des invariants mathematiques

3. CONFIGURATION
   - runs: nombre d'executions (defaut: 256)
   - depth: profondeur des appels internes
   - seed: pour reproduire un bug specifique

4. COMMANDE
   forge test --fuzz-runs 1000 --match-path test/Fuzz



---

## 2. Fuzz Test Simple

In [2]:
# Fuzz test basique
FUZZ_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SafeMath {
    // Fonction a tester
    function safeAdd(uint256 a, uint256 b) public pure returns (uint256) {
        uint256 c = a + b;
        require(c >= a, "Overflow");
        return c;
    }
    
    function safeMul(uint256 a, uint256 b) public pure returns (uint256) {
        if (a == 0) return 0;
        uint256 c = a * b;
        require(c / a == b, "Overflow");
        return c;
    }
}
'''

FUZZ_TEST_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SafeMath.sol";

contract SafeMathFuzzTest is Test {
    SafeMath sm;

    function setUp() public {
        sm = new SafeMath();
    }

    // Fuzz test: a et b sont generes aleatoirement
    function testFuzz_SafeAdd(uint256 a, uint256 b) public pure {
        // Si pas d'overflow, safeAdd doit reussir
        if (a + b >= a) {
            assertEq(sm.safeAdd(a, b), a + b);
        }
    }

    // Fuzz test avec contrainte
    function testFuzz_SafeMul(uint256 a, uint256 b) public pure {
        vm.assume(a > 0);  // Eviter division par zero
        vm.assume(b > 0);  // Cas non-trivial
        
        uint256 result = sm.safeMul(a, b);
        assertEq(result, a * b);
    }

    // Test avec bounds explicites
    function testFuzz_SafeAdd_Bounds() public pure {
        // Tester avec max uint256
        vm.assume(uint128(a) + uint128(b) == uint128(a) + uint128(b));
        assertEq(sm.safeAdd(a, b), a + b);
    }
}
'''

print("Contrat SafeMath:")
print(FUZZ_BASIC)
print("\n" + "="*50 + "\n")
print("Test Fuzz:")
print(FUZZ_TEST_BASIC)

Contrat SafeMath:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SafeMath {
    // Fonction a tester
    function safeAdd(uint256 a, uint256 b) public pure returns (uint256) {
        uint256 c = a + b;
        require(c >= a, "Overflow");
        return c;
    }

    function safeMul(uint256 a, uint256 b) public pure returns (uint256) {
        if (a == 0) return 0;
        uint256 c = a * b;
        require(c / a == b, "Overflow");
        return c;
    }
}



Test Fuzz:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SafeMath.sol";

contract SafeMathFuzzTest is Test {
    SafeMath sm;

    function setUp() public {
        sm = new SafeMath();
    }

    // Fuzz test: a et b sont generes aleatoirement
    function testFuzz_SafeAdd(uint256 a, uint256 b) public pure {
        // Si pas d'overflow, safeAdd doit reussir
        if (a + b >= a) {
            assertEq(sm.safeAdd(a, b), a + b);
        }
    }

   

---

## 3. Invariant Testing

Les invariants testent que des proprietes restent toujours vraies.

In [3]:
# Invariant testing pour un token ERC-20
INVARIANT_TEST = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SimpleToken.sol";

contract TokenInvariantTest is Test {
    SimpleToken token;
    address[] users;

    function setUp() public {
        token = new SimpleToken(1_000_000 * 10**18);
        
        // Creer des utilisateurs
        for (uint i = 0; i < 5; i++) {
            users.push(address(uint160(i + 1)));
        }
    }

    // Invariant: le total supply ne change jamais
    function invariant_TotalSupply() public view {
        uint256 initialSupply = token.totalSupply();
        assertEq(token.totalSupply(), initialSupply);
    }

    // Invariant: sum of balances == total supply
    function invariant_BalanceSum() public view {
        uint256 sum = 0;
        for (uint i = 0; i < users.length; i++) {
            sum += token.balanceOf(users[i]);
        }
        // Note: peut echouer si mint/burn existe
    }
}
'''

print("Test d'Invariants:")
print(INVARIANT_TEST)

Test d'Invariants:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SimpleToken.sol";

contract TokenInvariantTest is Test {
    SimpleToken token;
    address[] users;

    function setUp() public {
        token = new SimpleToken(1_000_000 * 10**18);

        // Creer des utilisateurs
        for (uint i = 0; i < 5; i++) {
            users.push(address(uint160(i + 1)));
        }
    }

    // Invariant: le total supply ne change jamais
    function invariant_TotalSupply() public view {
        uint256 initialSupply = token.totalSupply();
        assertEq(token.totalSupply(), initialSupply);
    }

    // Invariant: sum of balances == total supply
    function invariant_BalanceSum() public view {
        uint256 sum = 0;
        for (uint i = 0; i < users.length; i++) {
            sum += token.balanceOf(users[i]);
        }
        // Note: peut echouer si mint/burn existe
    }
}



---

## 4. vm.assume vs require

| Instruction | Usage | Effet |
|-------------|------|------|
| `vm.assume(cond)` | Dans les fuzz tests | Rejette l'input et genere une nouvelle |
| `require(cond, msg)` | Dans le contrat | Revert la transaction (echec du test) |

In [4]:
# Exemple avec vm.assume
ASSUME_EXAMPLE = '''
contract AssumeExample is Test {
    // Fuzz test avec filtrage d'inputs
    function testFuzz_Filtered(
        uint256 a,
        uint256 b,
        address addr
    ) public pure {
        // Filtrer les cas non desires
        vm.assume(a > 0);           // a non nul
        vm.assume(b > 0);           // b non nul
        vm.assume(a < type(uint128).max);  // a dans les limites
        vm.assume(addr != address(0));  // adresse non nulle
        vm.assume(addr.code.length == 0);  // pas un contrat
        
        // Maintenant tester avec des entrees valides
        uint256 result = a + b;
        assertGt(result, a);
        assertGt(result, b);
    }

    // Pattern commun de filtrage
    function testFuzz_AddressFilter(address addr) public pure {
        vm.assume(addr != address(0));
        vm.assume(addr != 0x0000000000000000000000000000000000000001);  // Ecrou
        // ... test logic
    }

    function testFuzz_UintRange(uint256 x) public pure {
        vm.assume(x >= 100 && x <= 1000);
        // x est maintenant dans [100, 1000]
    }
}
'''

print("Patterns avec vm.assume:")
print(ASSUME_EXAMPLE)

Patterns avec vm.assume:

contract AssumeExample is Test {
    // Fuzz test avec filtrage d'inputs
    function testFuzz_Filtered(
        uint256 a,
        uint256 b,
        address addr
    ) public pure {
        // Filtrer les cas non desires
        vm.assume(a > 0);           // a non nul
        vm.assume(b > 0);           // b non nul
        vm.assume(a < type(uint128).max);  // a dans les limites
        vm.assume(addr != address(0));  // adresse non nulle
        vm.assume(addr.code.length == 0);  // pas un contrat

        // Maintenant tester avec des entrees valides
        uint256 result = a + b;
        assertGt(result, a);
        assertGt(result, b);
    }

    // Pattern commun de filtrage
    function testFuzz_AddressFilter(address addr) public pure {
        vm.assume(addr != address(0));
        vm.assume(addr != 0x0000000000000000000000000000000000000001);  // Ecrou
        // ... test logic
    }

    function testFuzz_UintRange(uint256 x) public pure {
      

---

## 5. Configuration Fuzz

Dans `foundry.toml`:

In [5]:
# Configuration foundry.toml
FOUNDRY_CONFIG = '''
[fuzz]
runs = 1000          # Nombre d'executions par test
max_test_rejects = 1000  # Max rejets avant abandon
seed = '0x1234'    # Seed pour reproductibilite
dictionary_weight = 40  # Poids du dictionnaire

[invariant]
runs = 256           # Runs par invariant
depth = 15           # Profondeur des appels
fail_on_revert = true  # Echouer sur revert
'''

# Commandes
COMMANDS = '''
# Executer les fuzz tests
forge test --fuzz-runs 5000

# Executer uniquement les tests fuzz
forge test --match-path "test/*Fuzz*"

# Avec un seed specifique
forge test --fuzz-seed 0x3e8e

# Debug mode
forge test -vvvv --fuzz-runs 10
'''

print("Configuration foundry.toml:")
print(FOUNDRY_CONFIG)
print("\n" + "="*50 + "\n")
print("Commandes:")
print(COMMANDS)

Configuration foundry.toml:

[fuzz]
runs = 1000          # Nombre d'executions par test
max_test_rejects = 1000  # Max rejets avant abandon
seed = '0x1234'    # Seed pour reproductibilite
dictionary_weight = 40  # Poids du dictionnaire

[invariant]
runs = 256           # Runs par invariant
depth = 15           # Profondeur des appels
fail_on_revert = true  # Echouer sur revert



Commandes:

# Executer les fuzz tests
forge test --fuzz-runs 5000

# Executer uniquement les tests fuzz
forge test --match-path "test/*Fuzz*"

# Avec un seed specifique
forge test --fuzz-seed 0x3e8e

# Debug mode
forge test -vvvv --fuzz-runs 10



---

## 6. Exercices

In [6]:
# Exercice 1 : Fuzzer pour une pile LIFO
# TODO etudiant : ecrire des fuzz tests pour verifier les proprietes d'une pile LIFO
# Indice : quels invariants une pile LIFO doit-elle respecter ?
# Etape 1 : implementer testFuzz_PushPop - verifier que push puis pop laisse la taille inchangee
# Etape 2 : implementer testFuzz_LIFO - verifier que le dernier element empile est le premier depile

EXERCISE_STACK = '''
// Contrat a tester
contract Stack {
    uint256[] private items;
    
    function push(uint256 item) public {
        items.push(item);
    }
    
    function pop() public returns (uint256) {
        require(items.length > 0, "Empty stack");
        return items.pop();
    }
    
    function peek() public view returns (uint256) {
        require(items.length > 0, "Empty stack");
        return items[items.length - 1];
    }
    
    function size() public view returns (uint256) {
        return items.length;
    }
}

// Fuzz test
contract StackFuzzTest is Test {
    Stack stack;
    
    function setUp() public {
        stack = new Stack();
    }
    
    // Invariant: apres push puis pop, size est inchange
    function testFuzz_PushPop(uint256 value) public {
        // TODO: etudiant - verifier que push(value) incremente size de 1
        // TODO: etudiant - verifier que pop() retourne value
        // TODO: etudiant - verifier que size revient a la valeur initiale
    }
    
    // Invariant: LIFO property
    function testFuzz_LIFO(uint256[] memory values) public {
        // TODO: etudiant - filtrer les valeurs avec vm.assume (length > 0, <= 100)
        // TODO: etudiant - push tous les elements
        // TODO: etudiant - pop et verifier LIFO (dernier empile = premier depile)
        // Indice : parcourir le tableau a l'envers pour verifier
    }
}
'''
print("Exercice Stack Fuzz Test - a completer")
print("Implementez les tests fuzz marques TODO dans EXERCISE_STACK")


Exercice Stack Fuzz Test - a completer
Implementez les tests fuzz marques TODO dans EXERCISE_STACK


---

## 7. Resume

| Concept | Description |
|---------|-------------|
| Fuzz test | Test avec parametres aleatoires |
| `vm.assume` | Filtrer les entrees non desirees |
| Invariant | Propriete toujours vraie |
| `runs` | Nombre d'executions par test |

### Bonnes pratiques
1. Toujours limiter les ranges avec `vm.assume`
2. Tester les edge cases separement
3. Utiliser des seeds pour reproduire les bugs
4. Combiner fuzz tests et tests unitaires

---

**Notebook suivant** : [SC-14-Formal-Verification](SC-14-Formal-Verification.ipynb)

---

[<< Foundry Testing](SC-12-Foundry-Testing.ipynb) | [Formal Verification >>](SC-14-Formal-Verification.ipynb)